# Session Matching And Automated Recommendation Tool (SMART) 2026
This is the initial processing for AIM 2026

In [53]:
import importlib
import session_organizer
import pandas as pd
from google import genai
# Only needed if you want to reload the module after making changes
importlib.reload(session_organizer)

<module 'session_organizer' from 'c:\\Users\\jdv223\\OneDrive - University of Kentucky\\Programming\\AI\\SMART\\Session-Creation-Package\\session_organizer.py'>

# Process Steps

## Hybrid Sessions

### Invited Presentations

#### Load Invited Presentations from Hybrid Sessions

If you have existing hybrid sessions with pre-assigned presentations, you can load them.

In [54]:
# First, examine the Excel file structure
file_path = "Hybrid Session Invited Presentations.xlsx"
# Determine file type and read accordingly
if file_path.lower().endswith('.csv'):
    df_temp = pd.read_csv(file_path)
elif file_path.lower().endswith(('.xlsx', '.xls')):
    df_temp = pd.read_excel(file_path)
else:
    raise ValueError(f"Unsupported file format. Please use CSV (.csv) or Excel (.xlsx, .xls) files.")

print("Available columns:")
for i, col in enumerate(df_temp.columns):
    print(f"{i}: {col}")

print(f"\nFile contains {len(df_temp)} rows and {len(df_temp.columns)} columns")
print("\nFirst few rows preview:")
print(df_temp.head())

Available columns:
0: Session
1: Title
2: Abstract
3: Submission ID - 7 digits
4: Technical Community
5: Presenter: First Name
6: Presenter: Last Name

File contains 34 rows and 7 columns

First few rows preview:
                                             Session  \
0  Advancing Circular Bioeconomy Systems (CBS): O...   
1  Advancing Circular Bioeconomy Systems (CBS): O...   
2  Advancing Circular Bioeconomy Systems (CBS): O...   
3  Advancing Circular Bioeconomy Systems (CBS): O...   
4  Autonomous Machine Safety: Emerging Risks and ...   

                                               Title  \
0  BioCircular Valley: A Longitudinal Study and a...   
1  BioCircular Valley: A Longitudinal Study and a...   
2  From Wet Waste to Engineered Feedstocks: Advan...   
3  From Wet Waste to Engineered Feedstocks: Advan...   
4  Using a Layered Approach to Address Autonomous...   

                                            Abstract  \
0  BioCircular Valley (BioCirV) is a longitudinal...   
1

In [55]:
# Example: Load hybrid sessions from CSV/Excel file
hybrid_file_path = "Hybrid Session Invited Presentations.xlsx"  # Update this path as needed

# Load hybrid sessions using the flexible function
df_hybrid_presentations, df_hybrid_sessions, hybrid_session_col, title_col, abstract_col, abstract_id_col, topic_col = session_organizer.load_hybrid_sessions(
    hybrid_file_path,
    Session_column='Session',              # Actual column name in your file
    Title_column='Title',                  # Actual column name in your file  
    Abstract_column='Abstract',            # Actual column name in your file
    Abstract_ID_column='Submission ID - 7 digits',  # Actual column name in your file
    session_column='Session',              # Desired output column name
    title_column='Title',                  # Desired output column name
    abstract_column='Abstract',            # Desired output column name
    abstract_id_column='Abstract ID',      # Desired output column name
    topic_column='Title and Abstract'      # Combined column for embeddings
)

print(f"Loaded {len(df_hybrid_presentations)} hybrid presentations")
print(f"Session column: {hybrid_session_col}")
print(f"Title column: {title_col}")
print(f"Abstract column: {abstract_col}")
print(f"ID column: {abstract_id_col}")
print(f"Topic column: {topic_col}")

print("\nHybrid Sessions Summary:")
print(df_hybrid_sessions[[session_organizer.COLUMNS['CLUSTER_ID'], session_organizer.COLUMNS['SESSION_SIZE'], session_organizer.COLUMNS['HYBRID_SESSION_TITLE']]].to_string(index=False))

Loaded 24 hybrid presentations in 6 sessions
Session mapping: {'Advancing Circular Bioeconomy Systems (CBS): Opportunities and Challenges': 1, 'Autonomous Machine Safety: Emerging Risks and Protective Strategies': 2, 'Failed Research: Challenges and Opportunities': 3, 'Heat and Environmental Exposures: Impacts on Worker Safety': 4, 'Irrigation Management': 5, 'Remote Estimation of Evapotranspiration in Natural and Agricultural Systems': 6}
Loaded 24 hybrid presentations
Session column: Session
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract

Hybrid Sessions Summary:
 cluster_id  session_size                                                        hybrid_session_title
          1             1   Advancing Circular Bioeconomy Systems (CBS): Opportunities and Challenges
          2             5         Autonomous Machine Safety: Emerging Risks and Protective Strategies
          3             4                               Failed Rese

### Embed Invited Presentations from Hybrid Sessions

In [56]:
# Load environment variables
from dotenv import load_dotenv
import os
api_key = None  # Replace with your API key string if you want to provide it directly
load_dotenv(".env")
# Check for API key in environment variables if not provided
if api_key is None:
    if "GEMINI_API_KEY" not in os.environ:
        raise ValueError(
            "API key must be provided or in environmental variables. GEMINI_API_KEY not found in environment variables. Please set it in your .env file."
        )
    else:
        api_key = os.environ["GEMINI_API_KEY"]

# Validate API key
if not api_key:
    raise ValueError("API key is required to use Google GenAI.")

In [57]:
client = genai.Client()
print(f"Processing {len(df_hybrid_presentations)} presentations for embeddings.")
# Initialize the embedding column
df_hybrid_presentations['embedding'] = None

for idx, row in df_hybrid_presentations.iterrows():
    print(f"Processing presentation {idx + 1} of {len(df_hybrid_presentations)} (ID: {row[abstract_id_col]})")
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[row[topic_col]]
    )
    embedding = response.embeddings[0].values
    df_hybrid_presentations.at[idx, 'embedding'] = embedding

Processing 24 presentations for embeddings.
Processing presentation 1 of 24 (ID: 2600204.0)
Processing presentation 2 of 24 (ID: nan)
Processing presentation 3 of 24 (ID: nan)
Processing presentation 4 of 24 (ID: 2601296.0)
Processing presentation 5 of 24 (ID: nan)
Processing presentation 6 of 24 (ID: 2601405.0)
Processing presentation 7 of 24 (ID: nan)
Processing presentation 8 of 24 (ID: nan)
Processing presentation 9 of 24 (ID: nan)
Processing presentation 10 of 24 (ID: 2601149.0)
Processing presentation 11 of 24 (ID: nan)
Processing presentation 12 of 24 (ID: 2601419.0)
Processing presentation 13 of 24 (ID: 2601419.0)
Processing presentation 14 of 24 (ID: 2601419.0)
Processing presentation 15 of 24 (ID: 2601419.0)
Processing presentation 16 of 24 (ID: 2600302.0)
Processing presentation 17 of 24 (ID: 2600302.0)
Processing presentation 18 of 24 (ID: 2600870.0)
Processing presentation 19 of 24 (ID: 2600201.0)
Processing presentation 20 of 24 (ID: 2600437.0)
Processing presentation 21 

In [58]:
df_hybrid_presentations.to_parquet("hybrid presentations_with_embeddings.parquet", compression='snappy')
print("Embeddings saved to hybrid presentations_with_embeddings.parquet")

Embeddings saved to hybrid presentations_with_embeddings.parquet


## General Presentations

### Examine General Presentations

In [59]:
# First, examine the Excel file structure
file_path = "abstracts 1.20.26.xlsx"
df_temp = pd.read_excel(file_path)

print("Available columns:")
for i, col in enumerate(df_temp.columns):
    print(f"{i}: {col}")

print(f"\nFile contains {len(df_temp)} rows and {len(df_temp.columns)} columns")
print("\nFirst few rows preview:")
print(df_temp.head())

Available columns:
0: Submission ID
1: Submission Created Date & Time
2: External reference
3: Submission Completed Date & Time
4: Submission Status
5: Acceptance Status
6: # Reviews
7: Rating
8: Std Dev
9: Owner-E-mail Address
10: Owner-First Name
11: Owner-Last Name
12: Owner-Company/University
13: Owner-City
14: Owner-State
15: Owner-Country
16: Owner-CC Email
17: Owner-Test Profile
18: Submission-Call for Abstracts-Submission ID - 7 digits
19: Submission-Call for Abstracts-CHAIRS-enter your notes to other organizers or yourself here for session movement
20: Submission-Call for Abstracts-Select Your Session Preference
21: Submission-Call for Abstracts-Anticipated Session Topics
22: Submission-Call for Abstracts-Technical Community-First Preference 
23: Submission-Call for Abstracts-Technical Community-Second Preference 
24: Submission-Call for Abstracts-Presentation Title-Character max 160
25: Submission-Call for Abstracts-Abstract-Character max 4000-Abstracts will only be used to g

In [60]:
# Define your column selections based on the output above
TITLE_COLUMN = 'Submission-Call for Abstracts-Presentation Title-Character max 160'  
ABSTRACT_COLUMN = 'Submission-Call for Abstracts-Abstract-Character max 4000-Abstracts will only be used to group into topical sessions and evaluate quality of talk.'  
ID_COLUMN = 'Submission-Call for Abstracts-Submission ID - 7 digits'  



#### Load General Presentations

In [61]:
def load_presentations(file_path, Title_name='Title', Abstract_name='Abstract', Abstract_ID_name='Submission ID', 
                       title_column='Title', abstract_column='Abstract', abstract_id_column='Abstract ID', topic_column='Title and Abstract'):
    """
    Load presentations from an Excel file.
    Args:
        file_path (str): Path to the Excel file.
        Title_name (str): Spreadsheet column name that contains the titles.
        Abstract_name (str): Spreadsheet column name that contains the abstracts.
        Abstract_ID_name (str): Spreadsheet column name that contains the abstract IDs.
        title_column (str): Name for the title column in the output DataFrame.
        abstract_column (str): Name for the abstract column in the output DataFrame.
        abstract_id_column (str): Name for the abstract ID column in the output DataFrame.
        topic_column (str): Name for the combined title and abstract column in the output DataFrame.
    Returns:
        tuple: (df, title_column, abstract_column, abstract_id_column, topic_column)
    """
    # Determine file type and read accordingly
    if file_path.lower().endswith('.csv'):
        df = pd.read_csv(file_path)
    elif file_path.lower().endswith(('.xlsx', '.xls')):
        df = pd.read_excel(file_path)
    else:
        raise ValueError(f"Unsupported file format. Please use CSV (.csv) or Excel (.xlsx, .xls) files.")
    # Validate required columns exist in the file
    if Title_name not in df.columns or Abstract_name not in df.columns or Abstract_ID_name not in df.columns:
        raise ValueError(f"Columns '{Title_name}', '{Abstract_name}', '{Abstract_ID_name}' must be present in the Excel file.")
    
    # Create column rename map for the key columns
    column_rename_map = {
        Title_name: title_column,
        Abstract_name: abstract_column,
        Abstract_ID_name: abstract_id_column
    }
    
    # Rename the key columns while keeping all other columns
    df = df.rename(columns=column_rename_map)

    # Drop any presentations that are missing an abstract or title
    df = df.dropna(subset=[title_column, abstract_column])
    
    # Combine Titles and Abstracts with a colon in between. 
    # This should be the same as + but .agg() handles empty fields or fields that have non-text entries.
    df[topic_column] = df[[title_column, abstract_column]].agg(': '.join, axis=1)
    
    return df, title_column, abstract_column, abstract_id_column, topic_column

In [62]:
# Load the data using the session_organizer function
df, title_column, abstract_column, abstract_id_column, topic_column = load_presentations(
    file_path, 
    Title_name=TITLE_COLUMN,
    Abstract_name=ABSTRACT_COLUMN,
    Abstract_ID_name=ID_COLUMN
)

print(f"Loaded {len(df)} presentations successfully")
print(f"Title column: {title_column}")
print(f"Abstract column: {abstract_column}")
print(f"ID column: {abstract_id_column}")
print(f"Topic column: {topic_column}")

Loaded 1141 presentations successfully
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract


In [63]:

embedding_model = 'gemini-embedding-001'
print(f"Base model: {embedding_model}")
# Generate embeddings with resume capability
df_presentation_w_embeddings = session_organizer.embed_with_resume(df_presentations=df,
                                                                  topic_column=topic_column, 
                                                                  model_name=embedding_model, 
                                                                  embedding_model_name=None, 
                                                                  backup_filename="presentations_with_embeddings_backup.parquet", 
                                                                  delay_seconds=0)
print(f"Embeddings shape: {df_presentation_w_embeddings.shape}")


Base model: gemini-embedding-001
Presentations with embeddings loaded from presentations_with_embeddings_backup.parquet
Total presentations: 1141
Already embedded: 1141
Presentations to process: 0
Presentations with embeddings saved to presentations_with_embeddings_backup.parquet
Embeddings shape: (1141, 58)


## Save dataframe with embeddings

In [64]:
df_presentation_w_embeddings.to_parquet("presentations_with_embeddings.parquet", compression='snappy')
print("Embeddings saved to presentations_with_embeddings.parquet")

Embeddings saved to presentations_with_embeddings.parquet


df_presentation_w_embeddings = pd.read_parquet("presentations_with_embeddings.parquet")
print(f"Loaded {len(df_presentation_w_embeddings)} presentations with embeddings from parquet file.")

### Remove Duplicates and Near Duplicates
This only applies to the regular presentation list.

In [65]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
def remove_duplicates(df, similarity_func, threshold=0.95, embedding_column='embedding'):
    """
    Remove near-duplicate rows based on a similarity threshold.
    Args:
        df (pd.DataFrame): DataFrame containing presentation data with 'embedding' column.
        similarity_func (callable): Function to compute similarity between embeddings (e.g., cosine_similarity).
        threshold (float): Similarity threshold for considering items as near duplicates.
    Returns:
        pd.DataFrame: Presentations DataFrame with near-duplicate rows removed.
    """
    # Extract embeddings from the 'embedding' column into a numpy array
    embeddings_list = df[embedding_column].tolist()
    embeddings_array = np.array(embeddings_list)
    
    # Get the indices from the dataframe (important for referencing)
    presentation_indices = df.index.tolist()
    
    # Create a set to store the indices we want to REMOVE
    indices_to_remove = set()
    
    # Calculate the similarity matrix for the embeddings
    similarity_matrix = similarity_func(embeddings_array, embeddings_array)
    
    # Iterate through the upper triangle of the similarity matrix
    num_items = similarity_matrix.shape[0]

    for i in range(num_items):
        for j in range(i + 1, num_items):
            if similarity_matrix[i, j] >= threshold:
                # Get the actual DataFrame indices for positions i and j
                idx_i = presentation_indices[i]
                idx_j = presentation_indices[j]
                print(f"Near duplicate found: Index {idx_i} and Index {idx_j} (Similarity: {similarity_matrix[i, j]:.4f}).")
                # Remove the item with the LOWER index (keep the higher one)
                if idx_i < idx_j:
                    indices_to_remove.add(idx_i)
                else:
                    indices_to_remove.add(idx_j)

    # Convert the set of indices to remove into a list
    indices_to_remove_list = sorted(list(indices_to_remove))

    print(f"\nFound {len(indices_to_remove_list)} near-duplicate presentations to remove (keeping highest index).")
    print(f"Indices to remove: {indices_to_remove_list}")

    # --- Perform the removal ---
    df_cleaned = df.drop(index=indices_to_remove_list)

    # --- Verification ---
    print(f"\nFinal number of presentations: {len(df_cleaned)}")

    # Reset the index of the DataFrame to ensure it is clean and sequential
    df_cleaned = df_cleaned.reset_index(drop=True)
    
    return df_cleaned

In [66]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
similarity_threshold = 0.99
# Remove near-duplicate presentations based on the similarity threshold
df_cleaned = session_organizer.remove_duplicates(df_presentation_w_embeddings, similarity_func=cosine_similarity, embedding_column='embedding', threshold=similarity_threshold)

Near duplicate found: Index 410 and Index 950 (Similarity: 0.9932).
Near duplicate found: Index 412 and Index 525 (Similarity: 0.9908).

Found 2 near-duplicate presentations to remove (keeping highest index).
Indices to remove: [410, 412]

Final number of oral presentations: 1139


import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
similarity_threshold = 0.99
# Remove near-duplicate presentations based on the similarity threshold
df_cleaned = remove_duplicates(df_presentation_w_embeddings, similarity_func=cosine_similarity, threshold=similarity_threshold, embedding_column='embedding')

## Save Cleaned dataframe with embeddings

In [67]:
df_cleaned.to_parquet("cleaned_presentations_with_embeddings.parquet", compression='snappy')
print("Embeddings saved to cleaned_presentations_with_embeddings.parquet")

Embeddings saved to cleaned_presentations_with_embeddings.parquet


In [68]:
import pandas as pd
import numpy as np

df_cleaned = pd.read_parquet("cleaned_presentations_with_embeddings.parquet")
print(f"Loaded {len(df_cleaned)} presentations from cleaned_presentations_with_embeddings.parquet")
print(f"Columns: {df_cleaned.columns.tolist()}")
topic_column = "Title and Abstract"

Loaded 1139 presentations from cleaned_presentations_with_embeddings.parquet
Columns: ['Submission ID', 'Submission Created Date & Time', 'External reference', 'Submission Completed Date & Time', 'Submission Status', 'Acceptance Status', '# Reviews', 'Rating', 'Std Dev', 'Owner-E-mail Address', 'Owner-First Name', 'Owner-Last Name', 'Owner-Company/University', 'Owner-City', 'Owner-State', 'Owner-Country', 'Owner-CC Email', 'Owner-Test Profile', 'Abstract ID', 'Submission-Call for Abstracts-CHAIRS-enter your notes to other organizers or yourself here for session movement', 'Submission-Call for Abstracts-Select Your Session Preference', 'Submission-Call for Abstracts-Anticipated Session Topics', 'Submission-Call for Abstracts-Technical Community-First Preference ', 'Submission-Call for Abstracts-Technical Community-Second Preference ', 'Title', 'Abstract', 'Submission-Call for Abstracts-Session', 'Submission-Call for Abstracts-Order value (for HQ use)', 'Submission-Presentation Edits-Wou

In [69]:
oral_sel_text = 'Oral. I would like this submission to be considered for an Oral (standard or lightning) session.'
poster_sel_text = 'Poster. I would like to present this submission in a poster session.'
df_oral = df_cleaned[df_cleaned['Submission-Call for Abstracts-Select Your Session Preference'] == oral_sel_text].copy()
print(f"Number of oral presentations: {len(df_oral)}")

Number of oral presentations: 1139


In [70]:
# Hybrid Session Creation
session_column_name = 'Session Code'
# df_sessions, labels, metadata = session_organizer.create_sessions_w_hybrid(df, similarity_func=cosine_similarity, df_presentation_embeddings=df_presentation_embeddings,
#                                                                                      df_hybrid_presentations=df_hybrid_presentations,
#                                                                                      hybrid_session_column=hybrid_session_col, df_hybrid_embeddings=df_hybrid_embeddings,
#                                                                                      max_sessions=100, min_session_size=8, tree_merge_stop=1, cluster_column_name=session_column_name)
df_presentation_embeddings= session_organizer.extract_embeddings_dataframe(df_presentations=df_oral, 
                                                                           embedding_model_name=embedding_model, 
                                                                           embedding_column='embedding')
df_hybrid_embeddings= session_organizer.extract_embeddings_dataframe(df_presentations=df_hybrid_presentations, 
                                                                           embedding_model_name=embedding_model, 
                                                                           embedding_column='embedding')
df_sessions, labels, metadata = session_organizer.create_sessions_w_hybrid(df_presentations=df_oral, 
                                                                                             similarity_func=cosine_similarity, 
                                                                                             df_presentation_embeddings=df_presentation_embeddings,
                                                                                             df_hybrid_presentations=df_hybrid_presentations, 
                                                                                             hybrid_session_column=hybrid_session_col, 
                                                                                             df_hybrid_embeddings=df_hybrid_embeddings,
                                                                                             max_sessions=111, 
                                                                                             min_session_size=9,
                                                                                             tree_merge_stop=1, 
                                                                                             cluster_column_name="Session Code",
                                                                                             final_session_title_column=session_organizer.COLUMNS['FINAL_SESSION_TITLE'])

df_oral[session_column_name] = labels
print(f"Created {metadata['n_clusters']} sessions with {metadata['n_assigned_items']} presentations.")
print(f"Unassigned Presentations: {metadata['n_unassigned_items']}")
    

Created 111 sessions with 1139 presentations.
Unassigned Presentations: 0


In [71]:
def print_df_info(*dfs):
    local_vars = locals()
    for df in dfs:
        # Find variable name(s) pointing to this object
        var_names = [name for name, val in globals().items() if val is df]
        name_str = var_names[0] if var_names else "DataFrame"
        print(f"Available columns in {name_str}:")
        for i, col in enumerate(df.columns):
            print(f"{i}: {col}")
        print(f"\nDataframe contains {len(df)} rows and {len(df.columns)} columns\n")

# Usage:
print_df_info(df_oral, df_hybrid_presentations)

Available columns in df_oral:
0: Submission ID
1: Submission Created Date & Time
2: External reference
3: Submission Completed Date & Time
4: Submission Status
5: Acceptance Status
6: # Reviews
7: Rating
8: Std Dev
9: Owner-E-mail Address
10: Owner-First Name
11: Owner-Last Name
12: Owner-Company/University
13: Owner-City
14: Owner-State
15: Owner-Country
16: Owner-CC Email
17: Owner-Test Profile
18: Abstract ID
19: Submission-Call for Abstracts-CHAIRS-enter your notes to other organizers or yourself here for session movement
20: Submission-Call for Abstracts-Select Your Session Preference
21: Submission-Call for Abstracts-Anticipated Session Topics
22: Submission-Call for Abstracts-Technical Community-First Preference 
23: Submission-Call for Abstracts-Technical Community-Second Preference 
24: Title
25: Abstract
26: Submission-Call for Abstracts-Session
27: Submission-Call for Abstracts-Order value (for HQ use)
28: Submission-Presentation Edits-Would like to be moved to oral if avail

In [72]:
# First, define the column mapping lists before calling the function
df_columns_to_map = [
    'Submission-Call for Abstracts-Session',
    'Title', 
    'Abstract',
    'Abstract ID',
    'Submission-Call for Abstracts-Technical Community-First Preference ', 
    'Owner-First Name',
    'Owner-Last Name',
    'Title and Abstract',
    'Session Code',
    'embedding', 
    # Add more columns as needed - copy from print_df_info output above
]

hybrid_columns_to_map = [
    'Session',
    'Title', 
    'Abstract',
    'Abstract ID',
    'Technical Community', 
    'Presenter: First Name',
    'Presenter: Last Name',
    'Title and Abstract',
    'cluster_id',
    'embedding',
    # Add corresponding columns in same order
]
# Usage with manual mapping:
df_oral_with_hybrid, df_sessions_updated = session_organizer.add_hybrid_presentations_to_df(df=df_oral, 
                                                 df_hybrid_presentations=df_hybrid_presentations,
                                                 df_sessions=df_sessions,
                                                 session_column_name='Session Code',
                                                 hybrid_columns_to_map=hybrid_columns_to_map,
                                                 df_columns_to_map=df_columns_to_map,)

Column mapping: {'Session': 'Submission-Call for Abstracts-Session', 'Title': 'Title', 'Abstract': 'Abstract', 'Abstract ID': 'Abstract ID', 'Technical Community': 'Submission-Call for Abstracts-Technical Community-First Preference ', 'Presenter: First Name': 'Owner-First Name', 'Presenter: Last Name': 'Owner-Last Name', 'Title and Abstract': 'Title and Abstract', 'embedding': 'embedding'}
Added 1 hybrid presentations to session 0
New indices: [1139]
Added 5 hybrid presentations to session 1
New indices: [1140, 1141, 1142, 1143, 1144]
Added 4 hybrid presentations to session 2
New indices: [1145, 1146, 1147, 1148]
Added 5 hybrid presentations to session 3
New indices: [1149, 1150, 1151, 1152, 1153]
Added 4 hybrid presentations to session 4
New indices: [1154, 1155, 1156, 1157]
Added 5 hybrid presentations to session 5
New indices: [1158, 1159, 1160, 1161, 1162]


c:\Users\jdv223\OneDrive - University of Kentucky\Programming\AI\SMART\Session-Creation-Package\session_organizer.py:1716: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, mapped_hybrid_pres])
c:\Users\jdv223\OneDrive - University of Kentucky\Programming\AI\SMART\Session-Creation-Package\session_organizer.py:1716: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, mapped_hybrid_pres])
c:\Users\jdv223\OneDrive - University of Ken

In [75]:
df_presentation_w_hybrid_embeddings= session_organizer.extract_embeddings_dataframe(df_presentations=df_oral_with_hybrid, 
                                                                           embedding_model_name=embedding_model, 
                                                                           embedding_column='embedding')
embeddings_only = df_presentation_w_hybrid_embeddings.drop(columns=session_organizer.COLUMNS['EMBEDDING_MODEL'])
pres_similarities_matrix = cosine_similarity(embeddings_only.values, embeddings_only.values)

df_oral_with_hybrid['presentation_session_fit'],df_sessions_updated['session_coherence'], df_sessions_updated['session_distinctiveness'], df_session_session_similarity  = session_organizer.calculate_placement_metrics(
    df_presentations=df_oral_with_hybrid,
    df_sessions=df_sessions_updated,
    pres_similarities_matrix=pres_similarities_matrix,
    session_column_name=session_column_name
)

### Create Session Titles & Keywords

#### Gemini

In [76]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions_updated.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df_oral_with_hybrid, topic_column, model_name='gemini-2.5-flash-lite')


# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 7.0016 seconds
Average time per session: 2.3339 seconds
 cluster_id  session_size                           gen_presentation_indices hybrid_invited_presentations                                                       final_session_title  session_coherence  session_distinctiveness                                                                  Gemini Title 1                                                                Gemini Title 2                                                       Gemini Title 3                                                                                            Gemini Keywords
          0             9      [7, 252, 296, 300, 302, 304, 318, 1066, 1139]                          [0] Advancing Circular Bioeconomy Systems (CBS): Opportunities an

In [77]:

# Generate titles and keywords for all sessions using Gemini
df_sessions_updated = session_organizer.generate_session_titles_and_keywords(df_sessions_updated, df_oral_with_hybrid, topic_column, model_name='gemini-2.5-flash-lite')

# Display results
print(df_sessions_updated.head().to_string(index=False))

Processing session 0 (1/111)...
  ✓ Generated titles for session 0
Processing session 1 (2/111)...
  ✓ Generated titles for session 1
Processing session 2 (3/111)...
  ✓ Generated titles for session 2
Processing session 3 (4/111)...
  ✓ Generated titles for session 3
Processing session 4 (5/111)...
  ✓ Generated titles for session 4
Processing session 5 (6/111)...
  ✓ Generated titles for session 5
Processing session 6 (7/111)...
  ✓ Generated titles for session 6
Processing session 7 (8/111)...
  ✓ Generated titles for session 7
Processing session 8 (9/111)...
  ✓ Generated titles for session 8
Processing session 9 (10/111)...
  ✓ Generated titles for session 9
Processing session 10 (11/111)...
  ✓ Generated titles for session 10
Processing session 11 (12/111)...
  ✓ Generated titles for session 11
Processing session 12 (13/111)...
  ✓ Generated titles for session 12
Processing session 13 (14/111)...
  ✓ Generated titles for session 13
Processing session 14 (15/111)...
  ✓ Generated t

In [78]:
df_sessions_updated.to_csv("oral_sessions_w_hybrid_final.csv", index=False)
df_sessions_updated.to_parquet("oral_sessions_w_hybrid_final.parquet", index=False)
df_oral_with_hybrid.to_parquet("oral_presentations_w_hybrid_final.parquet", index=False)


### Include Hybrid Presentations?
Run the cell below to export the results with the hybrid invited presentation included

## Assign Committees to Sessions

In [79]:
# Read the committee file from CSV/Excel with flexible column selection
committee_file_path = 'ASABE Committees.csv'  # Update this path as needed (can also use .xlsx)

# Load committees
df_committees, committee_name_column, description_column, combined_column = session_organizer.load_committees(
    committee_file_path,
    Committee_Name_column='Committee_Name',  # Actual column name in your file
    Description_column='Description',        # Actual column name in your file
    committee_name_column='Committee_Name',  # Desired output column name
    description_column='Description',        # Desired output column name
    combined_column='Name_Description'       # Combined column for embeddings
)

print(f"Loaded {len(df_committees)} committees")
print(f"Committee name column: {committee_name_column}")
print(f"Description column: {description_column}")
print(f"Combined column: {combined_column}")
print("\nFirst few committees:")
print(df_committees[[committee_name_column, description_column]].head())

Loaded 108 committees
Committee name column: Committee_Name
Description column: Description
Combined column: Name_Description

First few committees:
                                      Committee_Name  \
0  ASE-09 Environmental Quality Coordinating Comm...   
1                          ASE-12 Forest Engineering   
2  ASE-134 Fertilizers, Soil Conditioners & US TA...   
3              ASE-16 Engineering for Sustainability   
4  ASE-347 and US TAG TC 347 Data-driven agrifood...   

                                         Description  
0  Leads and coordinates the activities of ASABE ...  
1  Forested landscapes are essential for clean wa...  
2  US Technical Advisory Group for ISO TC 134. Le...  
3  ASE-16 leads and coordinates ASABE activities ...  
4  Standardization in the field of big-picture, d...  


In [80]:

embedding_model = 'gemini-embedding-001'
print(f"Base model: {embedding_model}")
# Generate embeddings with resume capability
df_committees_w_embeddings = session_organizer.embed_with_resume(df_presentations=df_committees,
                                                                  topic_column=combined_column, 
                                                                  model_name=embedding_model, 
                                                                  embedding_model_name=None, 
                                                                  backup_filename="committees_with_embeddings_backup.parquet", 
                                                                  delay_seconds=0)
print(f"Embeddings shape: {df_committees_w_embeddings.shape}")


Base model: gemini-embedding-001
Presentations with embeddings loaded from committees_with_embeddings_backup.parquet
Total presentations: 108
Already embedded: 108
Presentations to process: 0
Presentations with embeddings saved to committees_with_embeddings_backup.parquet
Embeddings shape: (108, 4)


In [ ]:
# df_committee_embeddings = pd.DataFrame(df_committees['embedding'].tolist())
# df_committee_embeddings[DEFAULT_COLUMNS['EMBEDDING_MODEL']] = 'gemini-embedding-001'
# df_committee_embeddings.index = df_committees.index

In [81]:
df_presentation_w_hybrid_embeddings= session_organizer.extract_embeddings_dataframe(df_presentations=df_oral_with_hybrid, 
                                                                           embedding_model_name=embedding_model, 
                                                                           embedding_column='embedding')
df_committee_embeddings= session_organizer.extract_embeddings_dataframe(df_presentations=df_committees_w_embeddings, 
                                                                           embedding_model_name=embedding_model, 
                                                                           embedding_column='embedding')

In [82]:
# Find the most similar committees for each session
session_committee_matches = session_organizer.find_most_similar_committees_by_presentations(
    df_sessions_updated, 
    df_presentation_w_hybrid_embeddings, 
    df_committees, 
    df_committee_embeddings, 
    top_n=3,
)

Processing session 0 with 9 presentations...
Processing session 1 with 9 presentations...
Processing session 2 with 9 presentations...
Processing session 3 with 9 presentations...
Processing session 4 with 9 presentations...
Processing session 5 with 9 presentations...
Processing session 6 with 9 presentations...
Processing session 7 with 10 presentations...
Processing session 8 with 9 presentations...
Processing session 9 with 10 presentations...
Processing session 10 with 12 presentations...
Processing session 11 with 10 presentations...
Processing session 12 with 9 presentations...
Processing session 13 with 11 presentations...
Processing session 14 with 11 presentations...
Processing session 15 with 10 presentations...
Processing session 16 with 15 presentations...
Processing session 17 with 9 presentations...
Processing session 18 with 10 presentations...
Processing session 19 with 9 presentations...
Processing session 20 with 12 presentations...
Processing session 21 with 9 prese

In [83]:
df_sessions_updated = session_organizer.add_committee_matches_to_clusters(df_sessions_updated, session_committee_matches)
# Display a sample of the results
print(f"\nSample of top committee matches:")

print(df_sessions_updated.head(10).to_string(index=False))


Sample of top committee matches:
 cluster_id  session_size                            gen_presentation_indices hybrid_invited_presentations                                                         final_session_title  session_coherence  session_distinctiveness                                                            Gemini Title 1                                                                   Gemini Title 2                                                                  Gemini Title 3                                                                                             Gemini Keywords                                          Top Committee Match                                    2nd Committee Match                                    3rd Committee Match Top Committee Similarity 2nd Committee Similarity 3rd Committee Similarity
          0             9       [7, 252, 296, 300, 302, 304, 318, 1066, 1139]                          [0]   Advancing Circular Bioeconomy Systems (CB

### Include Hybrid Presentations?
Run the cell below to export the results with the hybrid invited presentation included

## Create sharable dataframe
Abstracts, names and emails should not be posted openly on the web. Remove them from the non-encrypted basic version of the data set.

First, check what columns are available. Then select the ones to remove.

In [84]:
print("Available columns:")
for i, col in enumerate(df_oral_with_hybrid.columns):
    print(f"{i}: {col}")

print(f"Dataframe contains {len(df_oral_with_hybrid)} rows and {len(df_oral_with_hybrid.columns)} columns")

Available columns:
0: Submission ID
1: Submission Created Date & Time
2: External reference
3: Submission Completed Date & Time
4: Submission Status
5: Acceptance Status
6: # Reviews
7: Rating
8: Std Dev
9: Owner-E-mail Address
10: Owner-First Name
11: Owner-Last Name
12: Owner-Company/University
13: Owner-City
14: Owner-State
15: Owner-Country
16: Owner-CC Email
17: Owner-Test Profile
18: Abstract ID
19: Submission-Call for Abstracts-CHAIRS-enter your notes to other organizers or yourself here for session movement
20: Submission-Call for Abstracts-Select Your Session Preference
21: Submission-Call for Abstracts-Anticipated Session Topics
22: Submission-Call for Abstracts-Technical Community-First Preference 
23: Submission-Call for Abstracts-Technical Community-Second Preference 
24: Title
25: Abstract
26: Submission-Call for Abstracts-Session
27: Submission-Call for Abstracts-Order value (for HQ use)
28: Submission-Presentation Edits-Would like to be moved to oral if available
29: Su

Copy and paste columns to remove in the list below.

In [85]:
columns_to_drop = [
    'Submission Status',
    'Acceptance Status',
    '# Reviews',
    'Rating',
    'Std Dev',
    'Owner-E-mail Address',
    'Owner-First Name',
    'Owner-Last Name',
    'Owner-Company/University',
    'Owner-City',
    'Owner-State',
    'Owner-Country',
    'Owner-CC Email',
    'Owner-Test Profile',
    'Submission-Call for Abstracts-CHAIRS-enter your notes to other organizers or yourself here for session movement',
    'Submission-Presentation Edits-1.  Please upload your paper from template - https://www.asabe.org/ManuscriptTemplates - doc or docx file',
    'Submission-Presentation Edits-2a.  Presenting Author First Name',
    'Submission-Presentation Edits-2b.  Presenting Author Last Name',
    'Submission-Presentation Edits-2c.  Presenting Author Affiliation',
    'Submission-Presentation Edits-2d.  Presenting Author\'s Location',
    'Submission-Presentation Edits-2e.  Presenting Author\'s Email',
    'Submission-Presentation Edits-3a.  Edited Presentation Title',
    'Submission-Presentation Edits-3b.  List all Authors',
    'Submission-Presentation Edits-4. Student Presentation?',
    'Applicant-E-mail Address',
    'Applicant-First Name',
    'Applicant-Last Name',
    'Applicant-Company/University',
    'Applicant-City',
    'Applicant-State',
    'Applicant-Country',
    'Applicant-CC Email',
    'Applicant-What is your ASABE membership number?  If you are not an ASABE member, please skip.',
    'Applicant-Address One',
    'Applicant-Address Two',
    'Applicant-Phone',
    'Applicant-Company/University.1',
    'Applicant-Title',
    'Applicant-Zip',
    'Applicant-Biography',
    'Applicant-Profile Photo',
    'Applicant-Applicant Type'
]
df_no_abstract = df_oral_with_hybrid.drop(columns_to_drop, axis=1, errors='ignore')

## Create Encrypted Dataframe

In [87]:
import os
from dotenv import load_dotenv
# Load environment variables
load_dotenv(".env")
if "DATAFRAME26_PW" not in os.environ:
    raise ValueError(
        "Dataframe Encryption Password must be in environmental variables. DATAFRAME26_PW not found in environment variables. Please set it in your .env file."
    )
else:
    password_df = os.environ["DATAFRAME26_PW"]
filename_enc_df = 'encrypted_dfAIM26.crypt'
# Create an encrypted dataframe for publishing.
import cryptpandas as crp
# Encrypt the DataFrame and save as a pickle
crp.to_encrypted(df_oral, password=password_df, path=filename_enc_df)

In [88]:
# Save the abstract-free DataFrame without encryption
df_no_abstract.to_parquet('df_no_abstractAIM26.parquet', compression='snappy')

In [89]:
# Save the Sessions Dataframe
df_sessions_updated.to_parquet('df_sessionsAIM26.parquet', compression='snappy')

## Save  Similarities
This is both the Presentation-Presentation Similarity Matrix and the Session-Session Similarity Matrix. The web app should not load the embedding model or run the similarity function for speed.

In [90]:
# Calculate the presentation similarity matrix
embeddings_only = df_presentation_w_hybrid_embeddings.drop(columns=session_organizer.COLUMNS['EMBEDDING_MODEL'])
pres_similarities_matrix = cosine_similarity(embeddings_only.values, embeddings_only.values)

# Convert to numpy if needed
pres_similarities_df = pd.DataFrame(pres_similarities_matrix, 
                                    index=df_presentation_w_hybrid_embeddings.index, 
                                    columns=df_presentation_w_hybrid_embeddings.index)
# Save presentation similarities matrix
pres_similarities_df.to_parquet('pres_similarities_matrixAIM26.parquet', compression='snappy')

# Save the session similiarities matrix DataFrame
df_session_session_similarity.to_parquet('session_similarities_matrixAIM26.parquet', compression='snappy')

print("✓ Similarity matrices saved as Parquet files")

✓ Similarity matrices saved as Parquet files


# Code to Save and Load Workspace

In [91]:
import pickle
from datetime import datetime
import inspect

# Create a timestamp for the filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"workspace_variables_{timestamp}.pkl"

# Get all variables except built-in ones and modules
variables_to_save = {}
excluded_vars = {}

for k, v in globals().items():
    # Skip private/magic variables
    if k.startswith('_'):
        continue
    # Skip Jupyter built-ins
    if k in ['In', 'Out', 'get_ipython', 'exit', 'quit']:
        continue
    # Skip functions and classes
    if callable(v) or inspect.isclass(v) or inspect.ismodule(v):
        continue
    
    # Try to pickle test - some objects may fail
    try:
        pickle.dumps(v)
        variables_to_save[k] = v
    except (pickle.PicklingError, TypeError) as e:
        excluded_vars[k] = type(v).__name__

# Save to pickle file
with open(filename, 'wb') as f:
    pickle.dump(variables_to_save, f)

print(f"Saved {len(variables_to_save)} variables to {filename}")
print(f"Variables saved: {list(variables_to_save.keys())}")
if excluded_vars:
    print(f"\nExcluded {len(excluded_vars)} variables that couldn't be serialized:")
    for var_name, var_type in excluded_vars.items():
        print(f"  - {var_name} ({var_type})")

Saved 69 variables to workspace_variables_20260121_163220.pkl
Variables saved: ['file_path', 'df_temp', 'i', 'col', 'hybrid_file_path', 'df_hybrid_presentations', 'df_hybrid_sessions', 'hybrid_session_col', 'title_col', 'abstract_col', 'abstract_id_col', 'topic_col', 'api_key', 'idx', 'row', 'response', 'embedding', 'TITLE_COLUMN', 'ABSTRACT_COLUMN', 'ID_COLUMN', 'df', 'title_column', 'abstract_column', 'abstract_id_column', 'topic_column', 'embedding_model', 'df_presentation_w_embeddings', 'similarity_threshold', 'df_cleaned', 'oral_sel_text', 'poster_sel_text', 'df_oral', 'session_column_name', 'df_presentation_embeddings', 'df_hybrid_embeddings', 'df_sessions', 'labels', 'metadata', 'embeddings_only', 'pres_similarities_matrix', 'df_session_session_similarity', 'timestamp', 'filename', 'variables_to_save', 'excluded_vars', 'k', 'v', 'var_name', 'var_type', 'df_sessions_sample', 'committee_file_path', 'df_committees', 'committee_name_column', 'description_column', 'combined_column', 

In [ ]:
import pickle

# Load the variables
with open(filename, 'rb') as f:
    loaded_vars = pickle.load(f)

# Restore to global namespace
globals().update(loaded_vars)

print(f"Loaded {len(loaded_vars)} variables")
print(f"Variables loaded: {list(loaded_vars.keys())}")